In [4]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    f1_score,
    classification_report,
)

DATA_DIR = "/content/drive/MyDrive/TUBerlin_Data"

# lstm --> needs it in (seq lenth, no. of features)

### Dataset
class ECGDataset(Dataset):
    def __init__(
        self,
        signals: np.ndarray,
        labels: np.ndarray,
    ):
        self.signals = torch.tensor(
            signals,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.signals) # --> no. of ecg signals

    def __getitem__(self, idx):
        x = self.signals[idx] # --> 1 complete ecg waveform (so 140 time steps)
        x = x.unsqueeze(-1) # feature dimension --> constrict to (140,1)

        # ECG class
        y = self.labels[idx] - 1

        return x, y

### CNN + BiLSTM Model with attention-style pooling
class CNNLSTMClassifier(nn.Module):
    def __init__(
        self,
        input_size: int,
        cnn_channels: list,
        kernel_size: int,
        hidden_size: int,
        num_layers: int,
        num_classes: int,
        dropout: float,
        bidirectional: bool = True,
    ):
        super().__init__()

        # --- 1D CNN feature extractor ---
        conv_layers = []
        in_channels = input_size

        for i, out_channels in enumerate(cnn_channels):
            conv_layers.append(
                nn.Conv1d(
                    in_channels=in_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=kernel_size // 2,
                )
            )
            conv_layers.append(nn.BatchNorm1d(out_channels))
            conv_layers.append(nn.ReLU())

            if i == 0:
                conv_layers.append(nn.MaxPool1d(kernel_size=2))

            in_channels = out_channels

        self.cnn = nn.Sequential(*conv_layers)

        # --- BiLSTM sequence model ---
        self.lstm = nn.LSTM(
            input_size=cnn_channels[-1],
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
            bidirectional=bidirectional,
        )

        lstm_out_dim = hidden_size * (2 if bidirectional else 1)

        self.dropout = nn.Dropout(dropout)

        # mean-pool + max-pool over time, concatenated -> richer summary
        # than just the final timestep's hidden state
        self.fc = nn.Sequential(
            nn.Linear(lstm_out_dim * 2, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)          # (batch, features, seq_len) for Conv1d
        x = self.cnn(x)
        x = x.permute(0, 2, 1)          # (batch, seq_len, channels) for LSTM

        lstm_output, _ = self.lstm(x)    # (batch, seq_len, lstm_out_dim)

        # Pool over the time dimension instead of only taking the last step
        mean_pooled = lstm_output.mean(dim=1)
        max_pooled, _ = lstm_output.max(dim=1)

        pooled = torch.cat([mean_pooled, max_pooled], dim=1)
        pooled = self.dropout(pooled)

        prediction = self.fc(pooled)

        return prediction

def plot_confusion_matrix(
    true_indices: np.ndarray, predicted_indices: np.ndarray
) -> None:
    class_names = [
        "I",
        "II",
        "III",
        "IV",
        "V",
    ]

    cm = confusion_matrix(
        true_indices,
        predicted_indices,
    )

    display = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )

    fig, ax = plt.subplots(figsize=(8, 6))

    display.plot(
        ax=ax,
        cmap="Blues",
        values_format="d",
    )

    plt.xticks(rotation=30, ha="right")
    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title("Confusion Matrix – ECG5000")
    plt.tight_layout()
    plt.show()

def main():

    # ============================================================
    # LOAD FOLD TRAINING DATA
    # ============================================================

    train = pd.read_csv(
        f"{DATA_DIR}/ECG5000_FOLD_1_TRAIN_3200.txt",
        sep=r"\s+",
        header=None,
    )

    X_train = train.iloc[:, 1:].values.astype(np.float32)
    y_train = train.iloc[:, 0].values.astype(np.int64)


    # ============================================================
    # LOAD FOLD VALIDATION DATA
    # ============================================================

    val = pd.read_csv(
        f"{DATA_DIR}/ECG5000_FOLD_1_VAL_800.txt",
        sep=r"\s+",
        header=None,
    )

    X_val = val.iloc[:, 1:].values.astype(np.float32)
    y_val = val.iloc[:, 0].values.astype(np.int64)


    print("Training:", X_train.shape)
    print("Validation:", X_val.shape)

    # Expected:
    # Training:   (3200, 140)
    # Validation: (800, 140)

    # ============================================================
    # NORMALIZE DATA
    # ============================================================

    def normalize(signals: np.ndarray) -> np.ndarray:
        mean = signals.mean(
            axis=1,
            keepdims=True,
        )

        std = signals.std(
            axis=1,
            keepdims=True,
        ) + 1e-8

        return (signals - mean) / std

    X_train = normalize(X_train)
    X_val = normalize(X_val)

    # ============================================================
    # CREATE DATASETS
    # ============================================================

    train_dataset = ECGDataset(
        signals=X_train,
        labels=y_train,
    )

    val_dataset = ECGDataset(
        signals=X_val,
        labels=y_val,
    )

    print("Number of training samples:", len(train_dataset))
    print("Number of validation samples:", len(val_dataset))

    x, y = train_dataset[0]

    print("ECG shape:", x.shape)
    print("Label:", y)

    # ============================================================
    # CLASS COUNTS / CLASS WEIGHTS
    # ============================================================

    num_classes = 5

    class_counts = np.array([
        np.sum(y_train == c)
        for c in range(1, num_classes + 1)
    ])

    class_weights = (
        len(y_train)
        / (num_classes * np.maximum(class_counts, 1))
    )

    class_weights = np.clip(
        class_weights,
        a_min=None,
        a_max=25.0,
    )

    class_weights = torch.tensor(
        class_weights,
        dtype=torch.float32,
    )

    print("Class counts (I-V):", class_counts)
    print("Class weights (I-V):", class_weights)

    # ============================================================
    # WEIGHTED RANDOM SAMPLER
    # ============================================================

    sample_weights = np.array([
        class_weights[int(label) - 1].item()
        for label in y_train
    ])

    sample_weights = torch.tensor(
        sample_weights,
        dtype=torch.double,
    )

    train_sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )

    # ============================================================
    # DATALOADERS
    # ============================================================

    batch_size = 32

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        sampler=train_sampler,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
    )

    x_batch, y_batch = next(iter(train_loader))

    print("Batch input shape:", x_batch.shape)
    print("Batch label shape:", y_batch.shape)

    ### Initialize model

    input_size = 1
    cnn_channels = [32, 64]
    kernel_size = 5
    hidden_size = 64
    num_layers = 1
    num_classes = 5

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    model = CNNLSTMClassifier(
        input_size=input_size,
        cnn_channels=cnn_channels,
        kernel_size=kernel_size,
        hidden_size=hidden_size,
        num_layers=num_layers,
        num_classes=num_classes,
        dropout=0.3,
        bidirectional=True,
    ).to(device)

    ### Parameter count (total vs trainable)
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters:     {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")

    class_weights = class_weights.to(device)

    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    outputs = model(x_batch)

    print("Model output shape:", outputs.shape)

    ### LOSS FUNCTION --> WEIGHTED CROSS ENTROPY LOSS
    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=0.05,  # slight smoothing helps with noisy medical labels
    )
    loss = criterion(
        outputs,
        y_batch,
    )

    print("Loss:", loss.item())

    ### Optimizer + LR scheduler
    learning_rate = 0.001
    weight_decay = 1e-4

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=5,
    )

    ### Training loop with early stopping
    epochs = 30
    patience = 15
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    best_model_state = None

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()

            outputs = model(x_batch)

            loss = criterion(
                outputs,
                y_batch,
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        ### Validation
        model.eval()

        val_loss = 0.0

        with torch.no_grad():

            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)

                outputs = model(x_batch)

                loss = criterion(
                    outputs,
                    y_batch,
                )

                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch {epoch + 1} | "
            f"Train Loss: {train_loss:.6f} | "
            f"Val Loss: {val_loss:.6f} | "
            f"LR: {current_lr:.6f}"
        )

        ### Early stopping on validation loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            best_model_state = {
                k: v.clone() for k, v in model.state_dict().items()
            }
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping at epoch {epoch + 1}")
                break

    ### Restore best model before evaluating
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    ### Evaluate on fold validation set
    model.eval()

    all_true = []
    all_preds = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch = x_batch.to(device)

            outputs = model(x_batch)

            predicted_classes = torch.argmax(outputs, dim=1)

            all_true.extend(y_batch.numpy())
            all_preds.extend(predicted_classes.cpu().numpy())

    all_true = np.array(all_true)
    all_preds = np.array(all_preds)

    # --------------------------------------------------------
    # Validation metrics
    # --------------------------------------------------------

    val_accuracy = (
        all_true == all_preds
    ).mean()

    balanced_acc = balanced_accuracy_score(
        all_true,
        all_preds,
    )

    macro_f1 = f1_score(
        all_true,
        all_preds,
        average="macro",
        zero_division=0,
    )

    print("\n============================")
    print("FOLD 1 VALIDATION RESULTS")
    print("============================")

    print(
        f"Accuracy:          "
        f"{val_accuracy * 100:.2f}%"
    )

    print(
        f"Balanced Accuracy: "
        f"{balanced_acc * 100:.2f}%"
    )

    print(
        f"Macro F1 Score:    "
        f"{macro_f1:.4f}"
    )

    ### Per-class breakdown - important for checking whether class V
    # (the rare one) is actually being learned, since aggregate
    # accuracy/F1 can look fine while V still gets ~0% recall.
    print("\nPer-class report:")
    print(
        classification_report(
            all_true,
            all_preds,
            target_names=["I", "II", "III", "IV", "V"],
            zero_division=0,
        )
    )

    plt.plot(
        train_losses,
        label="Training Loss"
    )

    plt.plot(
        val_losses,
        label="Validation Loss"
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("LSTM Training and Validation Loss")
    plt.legend()
    plt.show()

    plot_confusion_matrix(all_true, all_preds)

if __name__ == "__main__":
    main()


Training: (3200, 140)
Validation: (800, 140)
Class counts (I-V): [1868 1131   61  124   16]
Class weights (I-V): tensor([ 0.3426,  0.5659, 10.4918,  5.1613, 25.0000])


NameError: name 'normalize' is not defined